<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/bertopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertopic

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import os
import re

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b\d+\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
def build_topic_model(min_cluster_size=15):
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=(1,2),
        min_df=5
    )

    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics="auto",
        calculate_probabilities=True,
        verbose=True
    )

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/dataset"
TRANSCRIPT_EMB_DIR = "/content/drive/MyDrive/transcript_embeddings"

documents = []
embeddings = []
video_ids = []

for emb_fname in os.listdir(TRANSCRIPT_EMB_DIR):
    if not emb_fname.endswith(".npy"):
        continue

    video_id = emb_fname.replace(".npy", "")
    emb_path = os.path.join(TRANSCRIPT_EMB_DIR, emb_fname)
    txt_path = os.path.join(DATASET_DIR, video_id, "transcript.txt")

    if not os.path.exists(txt_path):
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if len(text.split()) < 5:
        continue

    text = preprocess_text(text)

    emb = np.load(emb_path)

    documents.append(text)
    embeddings.append(emb)
    video_ids.append(video_id)

embeddings = np.vstack(embeddings)

print(f"Loaded {len(documents)} transcripts with embeddings.")


In [ ]:
print(f"Embedding dim: {embeddings.shape[1]}")
print(f"Docs: {len(documents)}")


In [ ]:
topic_model = build_topic_model(min_cluster_size=15)
topics, probs = topic_model.fit_transform(documents, embeddings)


In [ ]:
df = pd.DataFrame({
    "video_id": video_ids,
    "document": documents,
    "topic": topics,
    "topic_prob": probs.max(axis=1)
})

df.to_csv("transcript_topics_new2.csv", index=False)


In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))


In [ ]:
for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    print(f"Topic {topic_id}: {', '.join([w for w,_ in words[:10]])}")


In [ ]:
for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    print(f"Topic {topic_id}: {', '.join([w for w,_ in words[:10]])}")

In [ ]:
#Topic 0
topic_id = 0
print("Top 3 representative transcripts:")
for doc in topic_model.get_representative_docs(topic_id)[:3]:
    print("-", doc[:200], "...")


In [ ]:
fig1 = topic_model.visualize_topics()
fig1.show()

fig2 = topic_model.visualize_barchart(top_n_topics=15)
fig2.show()

fig3 = topic_model.visualize_hierarchy()
fig3.show()


In [ ]:
topic_info[["Topic", "Count"]].sort_values("Count", ascending=False)


In [ ]:
outlier_rate = (df["topic"] == -1).mean()
print(f"Outlier proportion: {outlier_rate*100:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["topic_prob"], bins=30)
plt.xlabel("Topic assignment probability")
plt.ylabel("Number of transcripts")
plt.title("Topic Probability Distribution")
plt.show()


In [ ]:
sentiment_labels = "/content/drive/MyDrive/transcript_sentiment_0shot.csv"
df_sentiment = pd.read_csv(sentiment_labels)
print(df_sentiment.head())
print(df_sentiment.info())

In [ ]:

label_mapping = {"negative": -1, "neutral": 0, "positive": 1}
df_sentiment["sentiment_numeric"] = df_sentiment["label"].map(label_mapping)


In [ ]:
df_sentiment_topics = df.merge(df_sentiment, on="video_id", how="left")


print(df_sentiment_topics.head())



In [ ]:
# Example mapping
topic_labels = {
    0: "I/P Geo-Politics",
    1: "Police News",
    2: "U/R Geo-Politics",
    3: "UK Politics",
    4: "US inside politics",
    5: "General political talk",
    6: "US presidential news",
    7: "Day to day trade/economy news",
    8: "Local News",
    9: "Forecasts",
    10: "Talk show",
    11: "Trials / Legal Cases",
    12: "Local news",
    13: "Military, Borders Politics",
    14: "US external politics",
    15: "Fraud / Crime",
    16: "Epstein news",
    17: "empty",
    18: "Aircrafts and airports",
    19: "Energy and fuel news",
    20: "Europe news"
}

df["topic_label"] = df["topic"].map(topic_labels)
df_sentiment_topics = df.merge(df_sentiment, on="video_id", how="left")

print(df_sentiment_topics.columns)


In [ ]:
df_sentiment_topics.to_csv("transcript_topics_sentiment.csv", index=False)

In [ ]:
df_sentiment_topics.groupby("topic_label")["score"].mean().sort_values(ascending=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14,6))
sns.countplot(
    data=df_sentiment_topics,
    x="topic_label",
    hue="label",
    palette={"negative":"red", "neutral":"gray", "positive":"green"}
)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Number of transcripts")
plt.title("Sentiment distribution per topic")
plt.legend(title="Sentiment")
plt.show()



In [ ]:
avg_sentiment = df_sentiment_topics.groupby("topic_label")["sentiment_numeric"].mean().sort_values()

plt.figure(figsize=(14,6))
avg_sentiment.plot(kind="bar", color="skyblue")
plt.ylabel("Average sentiment (-1 negative, 0 neutral, 1 positive)")
plt.title("Average sentiment per topic")
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
topic_dists, _ = topic_model.approximate_distribution(documents)

print(f"Shape of topic distributions: {topic_dists.shape}")

In [ ]:
topic_dist_df = pd.DataFrame(
    topic_dists,
    columns=[f"t{i}" for i in range(topic_dists.shape[1])]
)

topic_dist_df.insert(0, "video_id", video_ids)

In [ ]:
topic_dist_df.to_csv("topic_distributions.csv", index=False)

In [ ]:
!pip install gensim

In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

def compute_coherence(topic_model, documents, coherence_type="c_v", top_n_words=10):

    # Tokenize documents
    tokenized_docs = [doc.split() for doc in documents]

    # Create dictionary
    dictionary = Dictionary(tokenized_docs)

    topics = []

    for topic_id in topic_model.get_topics().keys():
        if topic_id == -1:
            continue

        topic_words = topic_model.get_topic(topic_id)


        if topic_words is None or len(topic_words) == 0:
            continue

        words = [word for word, _ in topic_words[:top_n_words]]


        if len(words) == 0:
            continue
        if topic_id == 17:
            continue

        topics.append(words)

    if len(topics) == 0:
        raise ValueError("No valid topics found for coherence calculation.")

    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [ ]:
coherence_transcript = compute_coherence(
    topic_model,
    documents,
    coherence_type="c_v"
)

print("Transcript Model Coherence (c_v):", coherence_transcript)

In [ ]:
def topic_diversity(topic_model, top_n_words=10):
    topics = []
    for topic_id in topic_model.get_topics().keys():
        if topic_id == -1:
            continue
        words = [word for word, _ in topic_model.get_topic(topic_id)[:top_n_words]]
        topics.append(words)

    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)

    diversity = len(unique_words) / len(all_words)
    return diversity

In [ ]:
print("Transcript Topic Diversity:", topic_diversity(topic_model))


In [ ]:
from sklearn.metrics import silhouette_score


mask = np.array(topics) != -1

sil_score = silhouette_score(
    embeddings[mask],
    np.array(topics)[mask]
)

print("Silhouette Score:", sil_score)